# SPATIAL INTELLIGENCE - GRAPH METRICS & SHORTEST PATH

Compare graph structures using Minimum Spanning Tree, density, and diameter. Then compute the shortest navigation path across the prison floor plan.

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Configuration

In [ ]:
import os

renderer = "vscode"

# Paths
BASE_DIR = r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-02_RamonGarcia"
BREP_PATH = os.path.join(BASE_DIR, "geometries", "prison_clean.brep")
ASSETS_DIR = os.path.join(BASE_DIR, "assets")
os.makedirs(ASSETS_DIR, exist_ok=True)

GRID_SIZE = 2

# Image saving - the orchestrator sets this to True
SAVE_IMAGES = True

def save_fig(fig, filename):
    # Save a plotly figure to the assets directory as PNG.
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1600, height=1200, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Utility functions

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan and set up the analysis grid

In [ ]:
# Load the cleaned floor plan
floor_plan = Topology.ByBREPPath(BREP_PATH)

# Create a grid overlay
b_r = Wire.BoundingRectangle(floor_plan)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0, int(width) + GRID_SIZE, GRID_SIZE))
vRange = list(range(0, int(length) + GRID_SIZE, GRID_SIZE))
grid = Grid.EdgesByDistances(floor_plan, clip=True, uRange=uRange, vRange=vRange)

# Slice the floor plan to create a shell
shell = Topology.Slice(floor_plan, grid)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    f = Topology.SetDictionary(f, d)
print(f"Shell created with {len(faces)} faces")

## 6. Derive the navigation and analysis graphs

In [ ]:
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)
g_verts = Graph.Vertices(analysis_graph)
print(f"Analysis graph: {len(g_verts)} vertices, {len(Graph.Edges(analysis_graph))} edges")

## 7. Minimum Spanning Tree comparison

MST is computationally expensive on large grids, so we demonstrate it on a simple CellComplex to compare the original graph, its MST, and a complete graph.

In [ ]:
cc1 = CellComplex.Prism()
cc2 = Topology.Translate(cc1, 1.1, 0, 0)
cc3 = Topology.Translate(cc2, 1.1, 0, 0)
g1 = Graph.ByTopology(cc1)
g2 = Graph.ByTopology(cc2)
g3 = Graph.ByTopology(cc3)
g2 = Graph.MinimumSpanningTree(g2)
g3 = Graph.Complete(g3)

fig = Topology.Show(g1, g2, g3,
              vertexSize=12,
              vertexColor="red",
              edgeColor="lightgrey",
              edgeWidth=4,
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "05_mst_comparison.png")

## 8. Compare graph density and diameter

In [ ]:
for label, g in [("Original", g1), ("MST", g2), ("Complete", g3)]:
    dn = Graph.Density(g)
    dr = Graph.Diameter(g)
    print(f"{label:10s} | Density: {dn:.4f} | Diameter: {dr}")

## 9. Compute the shortest path across the floor plan

Using the navigation graph, find the shortest path between two corners of the prison. Then straighten it for a more realistic route.

In [ ]:
import time

start_vertex = Vertex.ByCoordinates(xmin + 2, ymax - 2, 0)
end_vertex = Vertex.ByCoordinates(xmax - 2, ymin + 2, 0)

crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)

t0 = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
print(f"Shortest path computed in {time.time() - t0:.2f}s")

t0 = time.time()
straight_path = Wire.Straighten(shortest_path, host=floor_plan)
print(f"Path straightened in {time.time() - t0:.2f}s")

print(f"Original path length:     {Wire.Length(shortest_path):.2f}")
print(f"Straightened path length: {Wire.Length(straight_path):.2f}")

# Style the paths
for edge in Topology.Edges(shortest_path):
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
for edge in Topology.Edges(straight_path):
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

## 10. Show the shortest path

In [ ]:
fig = Topology.Show(floor_plan, shortest_path, straight_path,
              camera=[0, 0, 6],
              faceColor=[210, 210, 250],
              faceOpacity=1,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "06_shortest_path.png")